# NASA C-MAPS FD001 - Exploratory Data Analysis

This notebook explores the turbofan engine degradation dataset and validates the data pipeline.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import CMAPSDataLoader
from feature_engineering import FeatureEngineer

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

## 1. Load Data

In [ ]:
loader = CMAPSDataLoader(data_dir='../data')

train_df = loader.load_train_data()
test_df, true_rul = loader.load_test_data()

print(f"Training set: {train_df.shape}")
print(f"Test set: {test_df.shape}")
print(f"True RUL values: {true_rul.shape}")

## 2. Data Overview

In [ ]:
train_df.head()

In [ ]:
train_df.info()

## 3. RUL Distribution

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(train_df['RUL'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Remaining Useful Life (cycles)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('RUL Distribution (Training Set)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(true_rul['RUL'], bins=30, edgecolor='black', alpha=0.7, color='orange')
plt.xlabel('Remaining Useful Life (cycles)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('RUL Distribution (Test Set)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Engine Lifecycle Analysis

In [ ]:
# Lifecycle length distribution
lifecycle_lengths = train_df.groupby('unit_id')['cycle'].max()

plt.figure(figsize=(10, 5))
plt.hist(lifecycle_lengths, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
plt.xlabel('Engine Lifecycle Length (cycles)', fontsize=12)
plt.ylabel('Number of Engines', fontsize=12)
plt.title('Distribution of Engine Lifecycles', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Average lifecycle: {lifecycle_lengths.mean():.1f} cycles")
print(f"Min lifecycle: {lifecycle_lengths.min()} cycles")
print(f"Max lifecycle: {lifecycle_lengths.max()} cycles")

## 5. Sample Engine Degradation

In [ ]:
# Plot RUL degradation for first 5 engines
plt.figure(figsize=(14, 6))

for engine_id in range(1, 6):
    engine_data = train_df[train_df['unit_id'] == engine_id]
    plt.plot(engine_data['cycle'], engine_data['RUL'], label=f'Engine {engine_id}', linewidth=2)

plt.xlabel('Cycle', fontsize=12)
plt.ylabel('Remaining Useful Life (cycles)', fontsize=12)
plt.title('RUL Degradation Curves - Sample Engines', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Sensor Correlation Analysis

In [ ]:
# Calculate sensor variances
sensor_cols = [col for col in train_df.columns if col.startswith('sensor_')]
sensor_variances = train_df[sensor_cols].var().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
plt.bar(range(len(sensor_variances)), sensor_variances.values, edgecolor='black', alpha=0.7)
plt.xticks(range(len(sensor_variances)), sensor_variances.index, rotation=45)
plt.xlabel('Sensor', fontsize=12)
plt.ylabel('Variance', fontsize=12)
plt.title('Sensor Variance Analysis (Informative vs Constant)', fontsize=14, fontweight='bold')
plt.axhline(y=0.01, color='red', linestyle='--', linewidth=2, label='Threshold (0.01)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nSensors with variance > 0.01: {(sensor_variances > 0.01).sum()}")
print(f"Sensors with variance <= 0.01: {(sensor_variances <= 0.01).sum()}")

## 7. Feature Engineering Preview

In [ ]:
engineer = FeatureEngineer()
train_processed = engineer.fit_transform(train_df)

print(f"Original features: {len([c for c in train_df.columns if c.startswith('sensor_')])}")
print(f"Engineered features: {len(engineer.get_feature_names())}")
print(f"\nFirst few engineered features:\n{engineer.get_feature_names()[:10]}")

## 8. Processed Data Sample

In [ ]:
train_processed.head(10)

## Summary

Key Insights:
- 100 training engines with varying lifecycles (128-356 cycles)
- RUL capped at 125 cycles (standard practice)
- ~7 sensors have near-zero variance (constant) → removed
- Rolling window features capture temporal degradation patterns
- Data ready for XGBoost training